In [56]:
import pandas as pd
import matplotlib.pyplot as plt

In [57]:
data = pd.read_csv("./data/training_data_lowercase.csv",sep='\t', names=['label', 'title'])
print(data.shape)
data.fillna("",inplace=True)
print(data.head())


(34152, 2)
   label                                              title
0      0  donald trump sends out embarrassing new year‚s...
1      0  drunk bragging trump staffer started russian c...
2      0  sheriff david clarke becomes an internet joke ...
3      0  trump is so obsessed he even has obama‚s name ...
4      0  pope francis just called out donald trump duri...


as part of pre proc we are able to see color codes in csv which are incorrectly interpretted by vs code
they are not color code but simply corresponds to episode num

we also see video/picture are there in some data points, while these are just metadata it could change the meaning of the sentence once we remove punctuations

we also see that this metadata in enclosed in () in Training sample while its enclosed in [] in testing data, so we might need different pre processing

In [58]:
from preProc import normalize_text

data["clean_text"] = data["title"].apply(normalize_text)
print(data.head)

<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  
0      donald trump sends out embarrassing new year s...  
1      drunk bragging trump staffer started rus

In [59]:
from preProc import remove_stopwords


data["no_stopwords"] = data["clean_text"].apply(remove_stopwords)
print(data.head)



<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  \
0      donald trump sends out embarrassing new year s...   
1      drunk bragging trump staffer started r

In [60]:
from preProc import tokens_lemm, tokens_stemm

data["stemmed"] = data["clean_text"].apply(tokens_stemm)
data["lemmed"] = data["clean_text"].apply(tokens_lemm)

print(data.head)


<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  \
0      donald trump sends out embarrassing new year s...   
1      drunk bragging trump staffer started r

In [61]:
from sklearn.model_selection import train_test_split
X=data.drop(columns=['label'])
y=data['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_val.shape}")
print(f"Training output size: {y_train.shape}")
print(f"Testing output size: {y_val.shape}")

Training set size: (27321, 5)
Testing set size: (6831, 5)
Training output size: (27321,)
Testing output size: (6831,)


* Use separate vectorizers, in model 1 we used the same vectorizer twice with fit_transform which verwrites vocabulary → inconsistent features
* Add raw cleaned text

BOW

In [62]:
from sklearn.feature_extraction.text import CountVectorizer
bow_vectorizer_lemm = CountVectorizer(max_features=5000)
X_train_BOW_L = bow_vectorizer_lemm.fit_transform(X_train["lemmed"])
X_val_BOW_L = bow_vectorizer_lemm.transform(X_val["lemmed"])

bow_vectorizer_stemm = CountVectorizer(max_features=5000)
X_train_BOW_S = bow_vectorizer_stemm.fit_transform(X_train["stemmed"])
X_val_BOW_S = bow_vectorizer_stemm.transform(X_val["stemmed"])

bow_vectorizer_raw = CountVectorizer(max_features=5000)
X_train_BOW_raw = bow_vectorizer_raw.fit_transform(X_train["clean_text"])
X_val_BOW_raw = bow_vectorizer_raw.transform(X_val["clean_text"])

TFIDF

In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_lemm = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),   
    min_df=3,
    max_df=0.85
)
X_train_TF_L = tfidf_lemm.fit_transform(X_train["lemmed"])
X_val_TF_L = tfidf_lemm.transform(X_val["lemmed"])

tfidf_stemm = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),   
    min_df=3,
    max_df=0.85
)
X_train_TF_S = tfidf_stemm.fit_transform(X_train["stemmed"])
X_val_TF_S = tfidf_stemm.transform(X_val["stemmed"])

tfidf_raw = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),   
    min_df=3,
    max_df=0.85
)
X_train_TF_raw = tfidf_raw.fit_transform(X_train["clean_text"])
X_val_TF_raw = tfidf_raw.transform(X_val["clean_text"])

In [64]:
from scipy.sparse import hstack, csr_matrix

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



experiments = {
    "BoW_stemm": (X_train_BOW_S, X_val_BOW_S),
    "BoW_lemm": (X_train_BOW_L, X_val_BOW_L),
    "BoW_row": (X_train_BOW_raw, X_val_BOW_raw),
    "TF-IDF_stemm": (X_train_TF_S, X_val_TF_S),
    "TF-IDF_lemm": (X_train_TF_L, X_val_TF_L),
    "TF-IDF_row": (X_train_TF_raw, X_val_TF_raw)
}

X_train_combined = hstack([X_train_TF_raw, X_train_TF_S])
X_val_combined = hstack([X_val_TF_raw, X_val_TF_S])

experiments["TF-IDF_combined"] = (X_train_combined, X_val_combined)

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVC": LinearSVC(),
}

results = []

for name, (Xtr, Xva) in experiments.items():
    for model_name, model in models.items():

        model.fit(Xtr, y_train)

        train_preds = model.predict(Xtr)
        val_preds = model.predict(Xva)

        train_acc = accuracy_score(y_train, train_preds)
        val_acc = accuracy_score(y_val, val_preds)

        # Loss calculation
        if model_name == "Linear SVC":
            train_scores = model.decision_function(Xtr)
            val_scores = model.decision_function(Xva)

            train_loss = hinge_loss(y_train, train_scores)
            val_loss = hinge_loss(y_val, val_scores)

        elif hasattr(model, "predict_proba"):
            train_probs = model.predict_proba(Xtr)
            val_probs = model.predict_proba(Xva)

            train_loss = log_loss(y_train, train_probs)
            val_loss = log_loss(y_val, val_probs)

        else:
            train_loss = None
            val_loss = None

        report = classification_report(y_val, val_preds, output_dict=True)

        precision = report["weighted avg"]["precision"]
        recall = report["weighted avg"]["recall"]
        f1 = report["weighted avg"]["f1-score"]

        results.append({
            "Text Vectorization": name,
            "Model": model_name,
            "Training Accuracy": train_acc,
            "Training Loss": train_loss,
            "Validation Accuracy": val_acc,
            "Validation Loss": val_loss,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1
        })

        print(
            f"{name} : {model_name} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}"
        )
results_df = pd.DataFrame(results).sort_values(
    by="Validation Accuracy",
    ascending=False
)

display(results_df)
results_df.to_csv("model_versioning_updated.csv", index=False)


BoW_stemm : Naive Bayes | Train Acc: 0.9424 | Train Loss: 0.1691 | Val Acc: 0.9340 | Val Loss: 0.1938
BoW_stemm : Logistic Regression | Train Acc: 0.9699 | Train Loss: 0.0999 | Val Acc: 0.9428 | Val Loss: 0.1409
BoW_stemm : Linear SVC | Train Acc: 0.9866 | Train Loss: 0.0791 | Val Acc: 0.9356 | Val Loss: 0.1655
BoW_lemm : Naive Bayes | Train Acc: 0.9444 | Train Loss: 0.1638 | Val Acc: 0.9366 | Val Loss: 0.1905
BoW_lemm : Logistic Regression | Train Acc: 0.9713 | Train Loss: 0.0987 | Val Acc: 0.9431 | Val Loss: 0.1421
BoW_lemm : Linear SVC | Train Acc: 0.9881 | Train Loss: 0.0759 | Val Acc: 0.9363 | Val Loss: 0.1639
BoW_row : Naive Bayes | Train Acc: 0.9451 | Train Loss: 0.1593 | Val Acc: 0.9409 | Val Loss: 0.1826
BoW_row : Logistic Regression | Train Acc: 0.9720 | Train Loss: 0.0989 | Val Acc: 0.9477 | Val Loss: 0.1406
BoW_row : Linear SVC | Train Acc: 0.9883 | Train Loss: 0.0745 | Val Acc: 0.9368 | Val Loss: 0.1650
TF-IDF_stemm : Naive Bayes | Train Acc: 0.9527 | Train Loss: 0.1542 | 

,Text Vectorization,Model,Training Accuracy,Training Loss,Validation Accuracy,Validation Loss,Precision,Recall,F1 Score
20,TF-IDF_combined,Linear SVC,0.998719,0.070969,0.953740,0.167473,0.953741,0.953740,0.953741
17,TF-IDF_row,Linear SVC,0.994070,0.110554,0.953008,0.187830,0.953039,0.953008,0.953013
11,TF-IDF_stemm,Linear SVC,0.993887,0.109980,0.953008,0.182788,0.953021,0.953008,0.953011
14,TF-IDF_lemm,Linear SVC,0.993924,0.109606,0.952423,0.184938,0.952421,0.952423,0.952422
19,TF-IDF_combined,Logistic Regression,0.974745,0.131080,0.951544,0.161041,0.951652,0.951544,0.951553
7,BoW_row,Logistic Regression,0.972036,0.098932,0.947738,0.140584,0.947923,0.947738,0.947750
13,TF-IDF_lemm,Logistic Regression,0.963252,0.174637,0.947153,0.193773,0.947307,0.947153,0.947164
10,TF-IDF_stemm,Logistic Regression,0.963362,0.173568,0.946714,0.191489,0.946962,0.946714,0.946726
16,TF-IDF_row,Logistic Regression,0.964423,0.176134,0.946274,0.195322,0.946493,0.946274,0.946287
4,BoW_lemm,Logistic Regression,0.971341,0.098727,0.943054,0.142129,0.943322,0.943054,0.943068
